# 🤖 AI Blog Article Generator
### 100% FREE — Powered by Google Gemini • Runs in Google Colab

**What you need:**
- ✅ A Google account (you already have one)
- ✅ A free Gemini API key → get it at **aistudio.google.com/apikey**

**What this does:**
1. Scrapes real news from the internet on your topic
2. Generates a 100% original AdSense-optimized article using Gemini AI
3. Saves HTML + Markdown + JSON files you can download

---
⚠️ **Run each cell one at a time using Shift+Enter. Wait for ✅ before moving to the next.**


## Step 1 — Install packages

In [1]:
!pip install -q fastapi uvicorn[standard] pydantic aiohttp beautifulsoup4 feedparser google-genai python-dotenv lxml nest-asyncio pyngrok requests
print("✅ All packages installed!")


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.5 MB/s eta 0:00:00
✅ All packages installed!


## Step 2 — Enter your FREE Gemini API key

> Get your key in 30 seconds:
> 1. Go to **aistudio.google.com/apikey**
> 2. Sign in with Google
> 3. Click **Create API key**
> 4. Copy it and paste below


In [15]:
import os
from getpass import getpass
from google import genai

print("Paste your Gemini API key (get it FREE at aistudio.google.com/apikey):")
api_key = getpass("GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = api_key

# Test the key immediately
print("\nTesting your key...")
try:
    client = genai.Client(api_key=api_key)
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents="Reply with exactly: KEY_WORKS"
    )
    if response.text:
        print(f"✅ Gemini API key is working!")
        print(f"   Model: gemini-2.0-flash")
    else:
        print("❌ Key accepted but no response. Try again.")
except Exception as e:
    print(f"❌ Key failed: {e}")
    print("   Make sure you copied the full key from aistudio.google.com/apikey")


Paste your Gemini API key (get it FREE at aistudio.google.com/apikey):
GEMINI_API_KEY: ··········

Testing your key...
❌ Key failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 0, model: gemini-2.0-flash\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_input_token_count, limit: 0, model: gemini-2.0-flash\nPlease retry in 14.059614741s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description':

## Step 3 — Create the API files

In [16]:
import os
os.makedirs("/content/blog_api", exist_ok=True)
print("📁 /content/blog_api/ created")


📁 /content/blog_api/ created


In [17]:
%%writefile /content/blog_api/generator.py
import os, json, re, math
from google import genai
from google.genai import types

class ArticleGenerator:
    def __init__(self):
        self.client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY", ""))
        self.model = "gemini-2.0-flash"

    async def generate(self, topic, news, tone="informative", word_count=1200, audience="general audience"):
        research = self._build_research(news)
        prompt = f"""You are a professional SEO content writer. Write 100% ORIGINAL blog articles.
Never copy from sources. Always synthesize into fresh original writing.
Follow Google AdSense quality guidelines.
Respond with valid JSON ONLY. No markdown fences. No text before or after the JSON.

TOPIC: {topic}
AUDIENCE: {audience}
TONE: {tone}
TARGET WORD COUNT: {word_count}

RESEARCH MATERIAL (reference only, do NOT copy):
{research}

Return this exact JSON:
{{
  "title": "Compelling SEO title 50-60 characters",
  "slug": "url-friendly-slug",
  "meta_description": "150-160 char meta description with keyword",
  "introduction": "2-3 engaging opening paragraphs",
  "sections": [
    {{
      "heading": "Section Heading",
      "level": 2,
      "content": "3-4 detailed paragraphs of original content",
      "key_points": ["Key point one", "Key point two", "Key point three"]
    }}
  ],
  "conclusion": "Strong 2-paragraph conclusion with call to action",
  "faq": [
    {{"question": "Question one?", "answer": "Detailed 2-3 sentence answer"}},
    {{"question": "Question two?", "answer": "Detailed answer"}},
    {{"question": "Question three?", "answer": "Detailed answer"}}
  ],
  "seo": {{
    "primary_keyword": "main keyword",
    "secondary_keywords": ["kw2", "kw3", "kw4", "kw5"],
    "meta_title": "Page Title | Blog",
    "focus_keyphrase": "exact focus phrase",
    "readability_score": "Good",
    "keyword_density": 1.5
  }},
  "adsense_tips": [
    {{"placement": "After introduction", "reason": "High viewability", "ad_type": "Display/Responsive"}},
    {{"placement": "Mid-content", "reason": "High engagement zone", "ad_type": "In-article"}},
    {{"placement": "Before conclusion", "reason": "High intent area", "ad_type": "Display/Responsive"}}
  ],
  "tags": ["tag1", "tag2", "tag3", "tag4"]
}}

Write at least {word_count} words total. Include 5-7 sections. 3 FAQ items."""

        response = self.client.models.generate_content(
            model=self.model,
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.8, max_output_tokens=4096)
        )
        raw = response.text.strip()
        data = self._parse_json(raw)
        data["html_content"] = self._build_html(data)
        data["markdown_content"] = self._build_markdown(data)
        data["word_count"] = self._count_words(data)
        data["reading_time"] = f"{math.ceil(data['word_count'] / 200)} min read"
        return data

    def _build_research(self, news):
        lines = []
        for i, a in enumerate(news.get("articles", [])[:6], 1):
            lines.append(f"[Source {i}] {a.get('publisher', '')} - {a.get('title', '')}")
            if a.get("summary"): lines.append(f"  {a['summary'][:350]}")
        for f in news.get("key_facts", [])[:8]: lines.append(f"• {f}")
        return "\n".join(lines) or f"Write about: {news.get('topic', '')}"

    def _build_html(self, d):
        p = [f'<h1>{self._e(d.get("title",""))}</h1>']
        if d.get("introduction"): p.append(f'<div class="intro">{self._ps(d["introduction"])}</div>')
        p.append("<!-- ADSENSE: After intro - Responsive Display Ad -->")
        for i, s in enumerate(d.get("sections", [])):
            lvl = s.get("level", 2)
            p.append(f'<h{lvl}>{self._e(s.get("heading",""))}</h{lvl}>')
            p.append(f'<div>{self._ps(s.get("content",""))}</div>')
            kps = s.get("key_points", [])
            if kps: p.append("<ul>" + "".join(f"<li>{self._e(k)}</li>" for k in kps) + "</ul>")
            if (i + 1) % 3 == 0: p.append(f"<!-- ADSENSE: Mid-content - In-Article Ad -->")
        p.append("<!-- ADSENSE: Before conclusion - Display Ad -->")
        if d.get("conclusion"): p.append(f'<h2>Conclusion</h2><div>{self._ps(d["conclusion"])}</div>')
        faq = d.get("faq", [])
        if faq:
            p.append('<h2>Frequently Asked Questions</h2>')
            for item in faq: p.append(f'<h3>{self._e(item.get("question",""))}</h3><p>{self._e(item.get("answer",""))}</p>')
        p.append("<!-- ADSENSE: After FAQ - Final Ad -->")
        return "\n".join(p)

    def _build_markdown(self, d):
        pts = [f"# {d.get('title','')}", f"> {d.get('meta_description','')}", "", d.get("introduction",""), ""]
        for s in d.get("sections", []):
            pts += [f"{'#'*s.get('level',2)} {s.get('heading','')}", s.get("content",""), ""]
            for kp in s.get("key_points", []): pts.append(f"- {kp}")
        pts += ["", "## Conclusion", d.get("conclusion",""), "", "## FAQ"]
        for item in d.get("faq", []): pts += [f"**{item.get('question','')}**", item.get("answer",""), ""]
        return "\n".join(pts)

    def _count_words(self, d):
        return len(" ".join([d.get("introduction",""), d.get("conclusion","")] + [s.get("content","") for s in d.get("sections",[])]).split())

    def _ps(self, t): return "".join(f"<p>{self._e(p.strip())}</p>" for p in t.split("\n") if p.strip())
    def _e(self, t): return str(t).replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    def _parse_json(self, text):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.MULTILINE)
        text = re.sub(r"\s*```\s*$", "", text, flags=re.MULTILINE).strip()
        try: return json.loads(text)
        except:
            m = re.search(r"\{.*\}", text, re.DOTALL)
            if m: return json.loads(m.group())
            raise ValueError(f"Cannot parse JSON from Gemini. Raw: {text[:300]}")


Overwriting /content/blog_api/generator.py


In [18]:
%%writefile /content/blog_api/scraper.py
import asyncio, aiohttp, feedparser, re, logging
from bs4 import BeautifulSoup
from urllib.parse import quote_plus
from datetime import datetime
logger = logging.getLogger(__name__)

class NewsScraper:
    HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    RSS_FEEDS = {
        "general": ["https://feeds.bbci.co.uk/news/rss.xml", "https://feeds.reuters.com/reuters/topNews"],
        "tech": ["https://feeds.feedburner.com/TechCrunch", "https://www.theverge.com/rss/index.xml"],
        "health": ["https://rss.medicalnewstoday.com/featurednews.xml"],
        "finance": ["https://www.cnbc.com/id/100003114/device/rss/rss.html"],
        "lifestyle": ["https://feeds.feedburner.com/huffingtonpost/wellness"],
    }
    def __init__(self): self.timeout = aiohttp.ClientTimeout(total=15)

    async def scrape(self, topic, category="general", max_sources=5):
        result = {"topic": topic, "articles": [], "sources": [], "key_facts": [], "scraped_at": datetime.utcnow().isoformat()}
        gathered = await asyncio.gather(self._google_news(topic, max_sources), self._rss_feeds(topic, category, max_sources), return_exceptions=True)
        for r in gathered:
            if isinstance(r, list): result["articles"].extend(r)
        result["articles"] = self._dedup(result["articles"])[:max_sources * 2]
        result["sources"] = [{"title": a.get("title",""), "url": a.get("url",""), "publisher": a.get("publisher","")} for a in result["articles"]]
        result["key_facts"] = self._facts(result["articles"])
        return result

    async def get_trending(self, category="general", limit=10):
        topics = []
        async with aiohttp.ClientSession(headers=self.HEADERS, timeout=self.timeout) as s:
            for url in self.RSS_FEEDS.get(category, self.RSS_FEEDS["general"])[:2]:
                try:
                    async with s.get(url) as r:
                        feed = feedparser.parse(await r.text())
                        for e in feed.entries[:5]: topics.append({"title": e.get("title",""), "url": e.get("link",""), "source": feed.feed.get("title","Unknown")})
                except Exception as e: logger.warning(f"RSS: {e}")
        return topics[:limit]

    async def _google_news(self, topic, max_results=5):
        arts = []
        try:
            async with aiohttp.ClientSession(headers=self.HEADERS, timeout=self.timeout) as s:
                async with s.get(f"https://news.google.com/rss/search?q={quote_plus(topic)}&hl=en-US&gl=US&ceid=US:en") as r:
                    feed = feedparser.parse(await r.text())
                    for e in feed.entries[:max_results]:
                        arts.append({"title": e.get("title",""), "url": e.get("link",""), "summary": BeautifulSoup(e.get("summary",""), "html.parser").get_text()[:400], "publisher": e.get("source",{}).get("title","Google News")})
        except Exception as e: logger.warning(f"GNews: {e}")
        return arts

    async def _rss_feeds(self, topic, category, max_sources=3):
        arts, kws = [], topic.lower().split()
        async with aiohttp.ClientSession(headers=self.HEADERS, timeout=self.timeout) as s:
            for url in self.RSS_FEEDS.get(category, self.RSS_FEEDS["general"]):
                try:
                    async with s.get(url) as r:
                        feed = feedparser.parse(await r.text())
                        for e in feed.entries:
                            score = sum(1 for k in kws if k in e.get("title","").lower() or k in e.get("summary","").lower())
                            if score < 1 and len(kws) > 1: continue
                            arts.append({"title": e.get("title",""), "url": e.get("link",""), "summary": BeautifulSoup(e.get("summary",""), "html.parser").get_text()[:400], "publisher": feed.feed.get("title","Unknown"), "score": score})
                except Exception as e: logger.warning(f"RSS {url}: {e}")
        arts.sort(key=lambda x: x.get("score",0), reverse=True)
        return arts[:max_sources]

    def _facts(self, articles):
        facts = []
        for a in articles:
            for s in re.split(r"(?<=[.!?])\s+", a.get("summary",""))[:5]:
                if 40 < len(s.strip()) < 300: facts.append(s.strip())
        return list(set(facts))[:15]

    def _dedup(self, arts):
        seen, out = set(), []
        for a in arts:
            k = re.sub(r"\W+","", a.get("title","").lower())[:50]
            if k and k not in seen: seen.add(k); out.append(a)
        return out


Overwriting /content/blog_api/scraper.py


In [19]:
%%writefile /content/blog_api/images.py
import aiohttp, os, logging
logger = logging.getLogger(__name__)

class ImageFinder:
    def __init__(self):
        self.unsplash = os.environ.get("UNSPLASH_ACCESS_KEY","")
        self.pexels = os.environ.get("PEXELS_API_KEY","")

    async def find(self, topic, count=3):
        queries = [topic, " ".join(topic.split()[:3])] + [topic] * count
        images = []
        async with aiohttp.ClientSession() as sess:
            for i, q in enumerate(queries[:count]):
                img = None
                if self.unsplash: img = await self._unsplash(sess, q)
                if not img and self.pexels: img = await self._pexels(sess, q)
                if not img: img = {"url": f"https://picsum.photos/seed/{i+1}/1200/630", "thumb": f"https://picsum.photos/seed/{i+1}/400/210", "alt": f"{q}", "caption": f"Image for: {q}", "source": "placeholder", "license": "Free placeholder"}
                images.append(img)
        return images

    async def _unsplash(self, s, q):
        try:
            async with s.get("https://api.unsplash.com/search/photos", params={"query": q, "per_page": 1, "orientation": "landscape"}, headers={"Authorization": f"Client-ID {self.unsplash}"}) as r:
                if r.status == 200:
                    d = await r.json()
                    res = d.get("results", [])
                    if res:
                        p = res[0]
                        return {"url": p["urls"]["regular"], "thumb": p["urls"]["thumb"], "alt": p.get("alt_description") or q, "caption": f"Photo by {p['user']['name']} on Unsplash", "source": "unsplash", "license": "Unsplash License"}
        except Exception as e: logger.warning(f"Unsplash: {e}")
        return None

    async def _pexels(self, s, q):
        try:
            async with s.get("https://api.pexels.com/v1/search", params={"query": q, "per_page": 1}, headers={"Authorization": self.pexels}) as r:
                if r.status == 200:
                    d = await r.json()
                    photos = d.get("photos", [])
                    if photos:
                        p = photos[0]
                        return {"url": p["src"]["large"], "thumb": p["src"]["medium"], "alt": p.get("alt") or q, "caption": f"Photo by {p['photographer']} on Pexels", "source": "pexels", "license": "Pexels License"}
        except Exception as e: logger.warning(f"Pexels: {e}")
        return None


Overwriting /content/blog_api/images.py


In [20]:
%%writefile /content/blog_api/main.py
from fastapi import FastAPI, HTTPException, BackgroundTasks
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from typing import Optional
import uuid, os
from datetime import datetime
from generator import ArticleGenerator
from scraper import NewsScraper
from images import ImageFinder
from google import genai

app = FastAPI(title="AI Blog Generator (Gemini FREE)", version="3.0.0")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])
jobs = {}
scraper, generator, image_finder = NewsScraper(), ArticleGenerator(), ImageFinder()

class ArticleRequest(BaseModel):
    topic: str
    category: Optional[str] = "general"
    tone: Optional[str] = "informative"
    word_count: Optional[int] = 1200
    audience: Optional[str] = "general audience"
    image_count: Optional[int] = 3

@app.get("/")
async def root(): return {"name": "AI Blog Generator (Gemini FREE)", "version": "3.0.0", "status": "running", "docs": "/docs"}

@app.get("/trending")
async def trending(category: str = "general", limit: int = 10):
    return {"topics": await scraper.get_trending(category, limit), "category": category}

@app.post("/generate")
async def gen_async(req: ArticleRequest, bg: BackgroundTasks):
    jid = str(uuid.uuid4())
    jobs[jid] = {"status": "pending", "step": "queued", "created_at": datetime.utcnow().isoformat()}
    bg.add_task(run_job, jid, req)
    return {"job_id": jid, "status": "pending", "poll_url": f"/status/{jid}", "result_url": f"/article/{jid}"}

@app.post("/generate/sync")
async def gen_sync(req: ArticleRequest): return await build(req)

@app.get("/status/{jid}")
async def status(jid: str):
    if jid not in jobs: raise HTTPException(404, "Job not found")
    return jobs[jid]

@app.get("/article/{jid}")
async def get_article(jid: str):
    if jid not in jobs: raise HTTPException(404, "Job not found")
    j = jobs[jid]
    if j["status"] == "failed": raise HTTPException(500, j.get("error"))
    if j["status"] != "completed": raise HTTPException(202, f"Not ready: {j.get('step')}")
    return j["article"]

async def run_job(jid, req):
    try:
        jobs[jid]["status"] = "processing"
        a = await build(req, jid)
        jobs[jid].update({"status": "completed", "article": a, "completed_at": datetime.utcnow().isoformat()})
    except Exception as e:
        jobs[jid].update({"status": "failed", "error": str(e)})

async def build(req, jid=None):
    def step(s):
        if jid and jid in jobs: jobs[jid]["step"] = s
    step("scraping news")
    news = await scraper.scrape(req.topic, req.category or "general", 5)
    step("generating with Gemini AI")
    article = await generator.generate(req.topic, news, req.tone or "informative", req.word_count or 1200, req.audience or "general audience")
    step("finding images")
    imgs = await image_finder.find(req.topic, req.image_count or 3)
    article.update({"images": imgs, "sources": news.get("sources",[]), "topic": req.topic, "category": req.category or "general", "generated_at": datetime.utcnow().isoformat(), "id": jid or str(uuid.uuid4())})
    return article


Overwriting /content/blog_api/main.py


## Step 4 — Start the server

In [21]:
import nest_asyncio, uvicorn, threading, sys, os, time, requests, subprocess

# Kill any old server first
subprocess.run(["pkill", "-f", "uvicorn"], capture_output=True)
time.sleep(1)

nest_asyncio.apply()

# Force fresh module load
for mod in list(sys.modules.keys()):
    if any(x in mod for x in ["generator","main","scraper","images"]):
        del sys.modules[mod]

sys.path.insert(0, "/content/blog_api")
os.chdir("/content/blog_api")

def run_server():
    uvicorn.run("main:app", host="0.0.0.0", port=8000, log_level="warning")

threading.Thread(target=run_server, daemon=True).start()
time.sleep(4)

try:
    r = requests.get("http://localhost:8000/", timeout=5)
    info = r.json()
    print(f"✅ Server running: {info['name']}")
    print(f"   Version: {info['version']}")
    print(f"   Status:  {info['status']}")
    print(f"\n📖 API docs: http://localhost:8000/docs")
except Exception as e:
    print(f"❌ Server error: {e}")
    print("   Try re-running this cell")


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


✅ Server running: AI Blog Generator (Gemini FREE)
   Version: 3.0.0
   Status:  running

📖 API docs: http://localhost:8000/docs


## Step 5 — Generate your article

> ✏️ **Change the TOPIC below to anything you want!**


In [22]:
import requests, json

# ✏️ CHANGE THIS TO YOUR TOPIC
TOPIC = "how to make money online with AI tools in 2025"
CATEGORY = "tech"     # general / tech / health / finance / lifestyle
WORD_COUNT = 1200     # minimum 800 for AdSense

print(f'🚀 Generating article: "{TOPIC}"')
print(f'   Category: {CATEGORY} | Words: {WORD_COUNT}')
print("⏳ Please wait 20-40 seconds...\n")

try:
    r = requests.post("http://localhost:8000/generate/sync", json={
        "topic": TOPIC,
        "category": CATEGORY,
        "tone": "informative",
        "word_count": WORD_COUNT,
        "audience": "general audience",
        "image_count": 3
    }, timeout=120)

    if r.status_code == 200:
        article = r.json()
        print("✅ Article generated successfully!\n")
        print(f"📰 Title:        {article['title']}")
        print(f"🔗 Slug:         {article['slug']}")
        print(f"📝 Word count:   {article['word_count']}")
        print(f"⏱️  Reading time: {article['reading_time']}")
        print(f"🔍 SEO keyword:  {article['seo'].get('primary_keyword','')}")
        print(f"📸 Images:       {len(article['images'])} found")
        print(f"🗞️  Sources:      {len(article['sources'])} scraped")
        print(f"\n💰 AdSense ad placements:")
        for tip in article["adsense_tips"]:
            print(f"   → {tip['placement']}: {tip['ad_type']}")
        print(f"\n📋 Meta description:")
        print(f"   {article['meta_description']}")
        print(f"\n🏷️  Tags: {', '.join(article.get('tags', []))}")
    else:
        print(f"❌ Error {r.status_code}: {r.text[:400]}")
        article = None

except requests.exceptions.Timeout:
    print("❌ Request timed out. Re-run this cell.")
    article = None
except Exception as e:
    print(f"❌ Error: {e}")
    article = None


🚀 Generating article: "how to make money online with AI tools in 2025"
   Category: tech | Words: 1200
⏳ Please wait 20-40 seconds...



ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

❌ Error 500: Internal Server Error


## Step 6 — Preview your article

In [23]:
from IPython.display import HTML, display

if not article:
    print("❌ No article yet. Run Step 5 first!")
else:
    styled = article["html_content"]\
        .replace("<!-- ADSENSE: After intro - Responsive Display Ad -->",
                 '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 15px;margin:20px 0;font-size:0.85em;color:#856404;border-radius:4px">📢 AD SLOT 1 — Responsive Display Ad (place your AdSense code here)</div>')\
        .replace("<!-- ADSENSE: Mid-content - In-Article Ad -->",
                 '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 15px;margin:20px 0;font-size:0.85em;color:#856404;border-radius:4px">📢 AD SLOT 2 — In-Article Ad (place your AdSense code here)</div>')\
        .replace("<!-- ADSENSE: Before conclusion - Display Ad -->",
                 '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 15px;margin:20px 0;font-size:0.85em;color:#856404;border-radius:4px">📢 AD SLOT 3 — Display Ad (place your AdSense code here)</div>')\
        .replace("<!-- ADSENSE: After FAQ - Final Ad -->",
                 '<div style="background:#fff3cd;border-left:4px solid #ffc107;padding:10px 15px;margin:20px 0;font-size:0.85em;color:#856404;border-radius:4px">📢 AD SLOT 4 — Final Ad (place your AdSense code here)</div>')

    css = "<style>h1{color:#1a1a2e;font-size:2em}h2{color:#16213e;border-bottom:3px solid #e94560;padding-bottom:8px;margin-top:30px}h3{color:#0f3460}p{color:#333;margin:12px 0}ul{background:#f8f9fa;padding:15px 30px;border-radius:8px}.intro{background:#f0f4ff;padding:20px;border-radius:8px}</style>"
    html_out = "<div style='font-family:Georgia,serif;max-width:800px;margin:auto;padding:20px;line-height:1.8'>" + css + styled + "</div>"
    display(HTML(html_out))


❌ No article yet. Run Step 5 first!


## Step 7 — Save and download your files

In [11]:
import json

if not article:
    print("❌ No article yet. Run Step 5 first!")
else:
    slug = article.get("slug", "my-article")

    # Save HTML (ready to publish)
    html_path = f"/content/{slug}.html"
    css = "<style>body{font-family:Georgia,serif;max-width:800px;margin:40px auto;padding:20px;line-height:1.8;color:#333}h1{color:#1a1a2e;font-size:2em}h2{color:#16213e;border-bottom:2px solid #e94560;padding-bottom:8px}h3{color:#0f3460}ul{background:#f8f9fa;padding:15px 30px;border-radius:8px}.intro{background:#f0f4ff;padding:20px;border-radius:8px;margin-bottom:20px}</style>"
    html_doc = "<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8"><meta name="viewport" content="width=device-width,initial-scale=1.0"><meta name="description" content="" + article["meta_description"] + ""><title>" + article["title"] + "</title>" + css + "</head><body>" + article["html_content"] + "</body></html>"
    with open(html_path, "w", encoding="utf-8") as f:
        f.write(html_doc)

    # Save Markdown (for WordPress / Ghost / Medium)
    md_path = f"/content/{slug}.md"
    with open(md_path, "w", encoding="utf-8") as f:
        f.write(article["markdown_content"])

    # Save full JSON (all data including SEO + AdSense tips)
    json_path = f"/content/{slug}.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(article, f, indent=2, ensure_ascii=False)

    print("✅ Files saved!")
    print(f"\n   📄 HTML (publish-ready): {html_path}")
    print(f"   📝 Markdown (WordPress): {md_path}")
    print(f"   🗂️  JSON (all data):      {json_path}")
    print()
    print("👉 To download:")
    print("   1. Click the 📁 Files icon in the LEFT SIDEBAR")
    print("   2. Right-click your file")
    print("   3. Click Download")
    print()
    print(f"📊 Article Summary:")
    print(f"   Words: {article['word_count']} | Reading time: {article['reading_time']}")
    print(f"   SEO keyword: {article['seo'].get('primary_keyword','')}")
    print(f"   AdSense slots: {len(article['adsense_tips'])}")


SyntaxError: invalid syntax (3240653974.py, line 11)

## Step 8 (Optional) — Get trending topics for your next article

In [ ]:
import requests

r = requests.get("http://localhost:8000/trending?category=tech&limit=8")
if r.status_code == 200:
    print("🔥 Trending topics right now (tech):\n")
    for i, topic in enumerate(r.json()["topics"], 1):
        print(f"  {i}. {topic['title']}")
        print(f"     Source: {topic['source']}\n")
else:
    print(f"Error: {r.text[:200]}")
